# Prompt Baseline

In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib/libnccl.so.2")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
nccl_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path + ":" + nccl_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


💻 Local Lab Server Environment Loaded.


## 1. Load Data & Base Model

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")

Loaded 2039 full-info validation samples and 2039 structural validation samples.


In [3]:
# === SELECT MODEL ===
MODEL_ID = "Nexusflow/NexusRaven-13B"

In [4]:
# === LOAD MODEL ===
COMPUTE_DTYPE = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

if MODEL_ID == "mistralai/Mistral-Nemo-Instruct-2407":
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											fix_mistral_regex=True
											)
else:
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											)
 
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)
model.eval()
print("Model loaded successfully!")

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded successfully!


## 2. Evaluate Baseline Full Information Prompt

In [5]:
# Evaluate on Full Info Dataset
acc_full, results_full = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_full,
    training_strategy="baseline",
    prompt_format="FullInfo",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating baseline - FullInfo:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Evaluating baseline - FullInfo:   1%|          | 20/2039 [00:26<43:50,  1.30s/it]

Evaluating baseline - FullInfo:   2%|▏         | 40/2039 [00:51<42:20,  1.27s/it]

...

Evaluating baseline - FullInfo:  97%|█████████▋| 1980/2039 [38:14<00:36,  1.63it/s]

Evaluating baseline - FullInfo:  98%|█████████▊| 2000/2039 [38:26<00:23,  1.64it/s]

Evaluating baseline - FullInfo:  99%|█████████▉| 2020/2039 [38:39<00:11,  1.63it/s]

Evaluating baseline - FullInfo: 100%|██████████| 2039/2039 [38:50<00:00,  1.14s/it]

\n--- Evaluation Results ---
Training Strategy: baseline
Prompt Format: FullInfo
Model: Nexusflow/NexusRaven-13B
Accuracy: 0.2109
Format Error Rate: 0.5439
Semantic Confusion: 0.2424
Option Bias (A): 0.8828
Latency: 2330.71 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_NexusRaven-13B_FullInfo_baseline.csv


## 3. Evaluate Baseline Structural-Only Prompt

In [6]:
# Evaluate on Structural Only Dataset
acc_struct, results_struct = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_struct,
    training_strategy="baseline",
    prompt_format="structOnly",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating baseline - structOnly:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Evaluating baseline - structOnly:   1%|          | 20/2039 [00:09<16:17,  2.06it/s]

Evaluating baseline - structOnly:   2%|▏         | 40/2039 [00:19<15:59,  2.08it/s]

...

Evaluating baseline - structOnly:  20%|█▉        | 400/2039 [03:11<13:09,  2.08it/s]

Evaluating baseline - structOnly:  21%|██        | 420/2039 [03:21<13:00,  2.07it/s]

Evaluating baseline - structOnly:  22%|██▏       | 440/2039 [03:30<12:46,  2.09it/s]

Evaluating baseline - structOnly:  23%|██▎       | 460/2039 [03:40<12:41,  2.07it/s]

Evaluating baseline - structOnly:  24%|██▎       | 480/2039 [03:50<12:32,  2.07it/s]